In [1]:
import gc
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'], check=False)
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
t0 = time.time()

DATA   = Path('/kaggle/input/competitions/march-machine-learning-mania-2026')
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)

CFG = {
    'elo_k_regular'   : 22,
    'elo_k_tourney'   : 32,
    'elo_init'        : 1500,
    'elo_home_bonus'  : 80,
    'elo_reversion'   : 0.33,
    'elo_margin_cap'  : 28,
    'massey_systems'  : ['POM','SAG','COL','DOL','MOR','WLK','RTH','BWE','DCI'],
    'massey_day'      : 133,
    'min_season'      : 2006,
    'val_seasons'     : [2023, 2024, 2025],
    'seed'            : 42,
    'recency_decay'   : 0.80,
    'clip_lo'         : 0.010,
    'clip_hi'         : 0.990,
    'power_confs'     : {'acc','big_ten','big_twelve','sec','big_east',
                         'pac_twelve','wcc','a_ten','mvc','wac'},
    'lgb_n_trials'    : 25,
    'xgb_n_trials'    : 20,
    'cat_n_trials'    : 30,
}

DIVIDER = '─' * 72
np.random.seed(CFG['seed'])
print(DIVIDER)
print('  March Machine Learning Mania 2026  —  Pipeline v3.0')
print(DIVIDER)


────────────────────────────────────────────────────────────────────────
  March Machine Learning Mania 2026  —  Pipeline v3.0
────────────────────────────────────────────────────────────────────────


In [2]:
print('\n[1/10] Loading data ...')

def load(fname):
    path = DATA / fname
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    return pd.read_csv(path)

m_reg_c  = load('MRegularSeasonCompactResults.csv')
w_reg_c  = load('WRegularSeasonCompactResults.csv')
m_tour_c = load('MNCAATourneyCompactResults.csv')
w_tour_c = load('WNCAATourneyCompactResults.csv')

M_REG_D_PATH = DATA / 'MRegularSeasonDetailedResults.csv'
W_REG_D_PATH = DATA / 'WRegularSeasonDetailedResults.csv'
M_TRN_D_PATH = DATA / 'MNCAATourneyDetailedResults.csv'
W_TRN_D_PATH = DATA / 'WNCAATourneyDetailedResults.csv'

m_seeds  = load('MNCAATourneySeeds.csv')
w_seeds  = load('WNCAATourneySeeds.csv')
m_teams  = load('MTeams.csv')
w_teams  = load('WTeams.csv')
m_conf   = load('MTeamConferences.csv')
w_conf   = load('WTeamConferences.csv')
massey   = load('MMasseyOrdinals.csv')
coaches  = load('MTeamCoaches.csv')
sub1     = load('SampleSubmissionStage1.csv')
sub2     = load('SampleSubmissionStage2.csv')

for _df in [sub1, sub2]:
    _df[['Season','T1','T2']] = _df['ID'].str.split('_', expand=True).astype(int)

print(f'   Men  regular : {len(m_reg_c):>7,} | Men  tourney: {len(m_tour_c):>5,}')
print(f'   Women regular: {len(w_reg_c):>7,} | Women tourney: {len(w_tour_c):>5,}')
print(f'   Massey entries: {len(massey):>7,}')
print(f'   Stage-1 matchups: {len(sub1):>7,} | Stage-2: {len(sub2):>7,}')



[1/10] Loading data ...
   Men  regular : 198,079 | Men  tourney: 2,585
   Women regular: 142,093 | Women tourney: 1,717
   Massey entries: 5,819,228
   Stage-1 matchups: 519,144 | Stage-2: 132,133


In [3]:
print(f'\n[2/10] Computing Elo ratings (with autocorrelation correction) ...')

def _expected(r_a, r_b):
    return 1.0 / (1.0 + 10.0 ** ((r_b - r_a) / 400.0))

def _margin_multiplier(margin, cap=CFG['elo_margin_cap']):
    margin = min(abs(margin), cap)
    return np.log(margin + 1) / np.log(cap + 1)

def compute_elo(compact_df, k_regular=CFG['elo_k_regular'],
                k_tourney=CFG['elo_k_tourney'], init=CFG['elo_init'],
                home_bonus=CFG['elo_home_bonus'], reversion=CFG['elo_reversion'],
                tourney_day_start=134):
    ratings = {}
    pre_snap, end_snap = [], []
    df = compact_df.sort_values(['Season','DayNum']).reset_index(drop=True)

    for season, grp in df.groupby('Season'):
        for tid in list(ratings):
            ratings[tid] = ratings[tid] + reversion * (init - ratings[tid])

        pre_saved = False
        for _, row in grp.iterrows():
            day        = row['DayNum']
            wid, lid   = row['WTeamID'], row['LTeamID']
            loc        = row.get('WLoc', 'N')
            score_diff = row['WScore'] - row['LScore']

            if day >= tourney_day_start and not pre_saved:
                for tid, elo in ratings.items():
                    pre_snap.append({'Season': season, 'TeamID': tid, 'EloPreTourney': elo})
                pre_saved = True

            if wid not in ratings: ratings[wid] = init
            if lid not in ratings: ratings[lid] = init

            r_w, r_l = ratings[wid], ratings[lid]
            if   loc == 'H': exp_w = _expected(r_w + home_bonus, r_l)
            elif loc == 'A': exp_w = _expected(r_w, r_l + home_bonus)
            else:            exp_w = _expected(r_w, r_l)

            k_adj        = (k_tourney if day >= tourney_day_start else k_regular) * _margin_multiplier(score_diff)
            delta        = k_adj * (1.0 - exp_w)
            ratings[wid] = r_w + delta
            ratings[lid] = r_l - delta

        if not pre_saved:
            for tid, elo in ratings.items():
                pre_snap.append({'Season': season, 'TeamID': tid, 'EloPreTourney': elo})

        for tid, elo in ratings.items():
            end_snap.append({'Season': season, 'TeamID': tid, 'EloEndSeason': elo})

    pre_df  = pd.DataFrame(pre_snap)
    end_df  = pd.DataFrame(end_snap)
    merged  = pre_df.merge(end_df, on=['Season','TeamID'], how='outer')
    merged['EloPreTourney'] = merged['EloPreTourney'].fillna(merged['EloEndSeason'])
    return merged

def compute_elo_offense_defense(compact_df, init=1500.0, k=20.0,
                                 reversion=0.33, tourney_day_start=134):
    o_rat = {}
    d_rat = {}
    pre_o, pre_d = [], []

    df = compact_df.sort_values(['Season','DayNum']).reset_index(drop=True)

    for season, grp in df.groupby('Season'):
        for tid in list(o_rat):
            o_rat[tid] = o_rat[tid] + reversion * (init - o_rat[tid])
            d_rat[tid] = d_rat[tid] + reversion * (init - d_rat[tid])

        pre_saved = False
        for _, row in grp.iterrows():
            day      = row['DayNum']
            wid, lid = row['WTeamID'], row['LTeamID']
            ws, ls   = float(row['WScore']), float(row['LScore'])

            if day >= tourney_day_start and not pre_saved:
                for tid in list(o_rat):
                    pre_o.append({'Season': season, 'TeamID': tid, 'OffRtg': o_rat[tid]})
                    pre_d.append({'Season': season, 'TeamID': tid, 'DefRtg': d_rat[tid]})
                pre_saved = True

            for tid in [wid, lid]:
                if tid not in o_rat: o_rat[tid] = init
                if tid not in d_rat: d_rat[tid] = init

            exp_ws = init + (o_rat[wid] - d_rat[lid])
            exp_ls = init + (o_rat[lid] - d_rat[wid])

            o_rat[wid] += k * (ws - exp_ws) / 100
            d_rat[lid] += k * (ls - exp_ls) / 100
            o_rat[lid] += k * (ls - exp_ls) / 100
            d_rat[wid] += k * (ws - exp_ws) / 100

        if not pre_saved:
            for tid in list(o_rat):
                pre_o.append({'Season': season, 'TeamID': tid, 'OffRtg': o_rat[tid]})
                pre_d.append({'Season': season, 'TeamID': tid, 'DefRtg': d_rat[tid]})

    o_df = pd.DataFrame(pre_o)
    d_df = pd.DataFrame(pre_d)
    return o_df.merge(d_df, on=['Season','TeamID'])

m_all_c  = pd.concat([m_reg_c, m_tour_c], ignore_index=True)
w_all_c  = pd.concat([w_reg_c, w_tour_c], ignore_index=True)

m_elo_df = compute_elo(m_all_c)
w_elo_df = compute_elo(w_all_c)
m_od_df  = compute_elo_offense_defense(m_all_c)
w_od_df  = compute_elo_offense_defense(w_all_c)

m_elo_lu  = m_elo_df.set_index(['Season','TeamID'])['EloPreTourney'].to_dict()
w_elo_lu  = w_elo_df.set_index(['Season','TeamID'])['EloPreTourney'].to_dict()
m_off_lu  = m_od_df.set_index(['Season','TeamID'])['OffRtg'].to_dict()
m_def_lu  = m_od_df.set_index(['Season','TeamID'])['DefRtg'].to_dict()
w_off_lu  = w_od_df.set_index(['Season','TeamID'])['OffRtg'].to_dict()
w_def_lu  = w_od_df.set_index(['Season','TeamID'])['DefRtg'].to_dict()

print(f'   Men  Elo: {len(m_elo_df):,} team-seasons')
print(f'   Women Elo: {len(w_elo_df):,} team-seasons')
print(f'   Off/Def ratings computed for both genders')



[2/10] Computing Elo ratings (with autocorrelation correction) ...
   Men  Elo: 14,206 team-seasons
   Women Elo: 9,952 team-seasons
   Off/Def ratings computed for both genders


In [4]:
print(f'\n[3/10] Building Massey Ordinals features ...')

TARGET_SYSTEMS = CFG['massey_systems']

def build_massey_features(massey_df):
    day_limit = CFG['massey_day']
    pre  = massey_df[massey_df['RankingDayNum'] <= day_limit].copy()
    idx  = pre.groupby(['Season','SystemName','TeamID'])['RankingDayNum'].idxmax()
    pre  = pre.loc[idx.values].reset_index(drop=True)
    avail_systems = [s for s in TARGET_SYSTEMS if s in pre['SystemName'].values]
    pre  = pre[pre['SystemName'].isin(avail_systems)]

    pivot = pre.pivot_table(
        index=['Season','TeamID'], columns='SystemName',
        values='OrdinalRank', aggfunc='first'
    ).reset_index()
    pivot.columns.name = None
    sys_cols = [c for c in avail_systems if c in pivot.columns]

    for sc in sys_cols:
        pivot[sc] = pivot.groupby('Season')[sc].transform(
            lambda x: (x.max() - x + 1) / x.count()
        )
        pivot.rename(columns={sc: f'mas_{sc}'}, inplace=True)

    pct_cols = [f'mas_{s}' for s in sys_cols if f'mas_{s}' in pivot.columns]
    pivot['mas_mean']    = pivot[pct_cols].mean(axis=1)
    pivot['mas_min']     = pivot[pct_cols].min(axis=1)
    pivot['mas_max']     = pivot[pct_cols].max(axis=1)
    pivot['mas_std']     = pivot[pct_cols].std(axis=1)
    pivot['mas_n_sys']   = pivot[pct_cols].notna().sum(axis=1)
    pivot['mas_top3_mean'] = pivot[pct_cols].apply(
        lambda r: r.nlargest(3).mean() if r.notna().sum() >= 3 else r.mean(), axis=1)

    print(f'   Massey: {len(pivot):,} team-seasons | systems used: {sys_cols}')
    return pivot

massey_feats = build_massey_features(massey)
massey_lu    = massey_feats.set_index(['Season','TeamID']).to_dict('index')
MAS_COLS     = [c for c in massey_feats.columns if c not in ('Season','TeamID')]

del massey_feats, massey
gc.collect()



[3/10] Building Massey Ordinals features ...
   Massey: 8,355 team-seasons | systems used: ['POM', 'SAG', 'COL', 'DOL', 'MOR', 'WLK', 'RTH', 'BWE', 'DCI']


0

In [5]:
print(f'\n[4/10] Computing Four Factors & box-score features ...')

def compute_four_factors(detail_path):
    need = ['Season','DayNum',
            'WTeamID','WScore','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA',
            'WOR','WDR','WAst','WTO','WStl','WBlk',
            'LTeamID','LScore','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA',
            'LOR','LDR','LAst','LTO','LStl','LBlk']

    all_cols = pd.read_csv(detail_path, nrows=0).columns.tolist()
    use_cols = [c for c in need if c in all_cols]

    df = pd.read_csv(detail_path, usecols=use_cols,
                     dtype={c: 'float32' for c in use_cols
                            if c not in ('Season','DayNum','WTeamID','LTeamID')})
    df = df[df['DayNum'] <= 132].copy()

    def _side(team_col, score_col, allow_col, opp_dr_col, win_val,
              fgm, fga, fgm3, fga3, ftm, fta, orb, dr, ast, to, stl, blk):
        d = {
            'Season': df['Season'], 'TeamID': df[team_col], 'Win': win_val,
            'Score': df[score_col], 'Allowed': df[allow_col],
            'FGM': df[fgm], 'FGA': df[fga], 'FGM3': df[fgm3], 'FGA3': df[fga3],
            'FTM': df[ftm], 'FTA': df[fta],
            'OR': df[orb], 'DR': df[dr], 'OppDR': df[opp_dr_col],
            'Ast': df[ast], 'TO': df[to],
        }
        if stl in df.columns: d['Stl'] = df[stl]
        if blk in df.columns: d['Blk'] = df[blk]
        return pd.DataFrame(d)

    w_side = _side('WTeamID','WScore','LScore','LDR', 1,
                   'WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA',
                   'WOR','WDR','WAst','WTO','WStl','WBlk')
    l_side = _side('LTeamID','LScore','WScore','WDR', 0,
                   'LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA',
                   'LOR','LDR','LAst','LTO','LStl','LBlk')

    del df; gc.collect()
    g = pd.concat([w_side, l_side], ignore_index=True)
    del w_side, l_side; gc.collect()

    sum_cols = ['Win','Score','Allowed','FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','OppDR','Ast','TO']
    for c in ['Stl','Blk']:
        if c in g.columns: sum_cols.append(c)

    agg_dict = {c: (c, 'sum') for c in sum_cols}
    agg_dict['Games'] = ('Win', 'count')
    agg = g.groupby(['Season','TeamID']).agg(**agg_dict).reset_index()

    eps_g = 1e-6
    g['Margin']    = g['Score'] - g['Allowed']
    g['FTRateG']   = g['FTM'] / (g['FGA'] + eps_g)
    g['OffPossG']  = g['FGA'] - g['OR'] + g['TO'] + 0.44 * g['FTA']

    std_agg = g.groupby(['Season','TeamID']).agg(
        MarginStd  = ('Margin',  'std'),
        FTRateStd  = ('FTRateG', 'std'),
        PossStd    = ('OffPossG','std'),
    ).reset_index().fillna(0)
    agg = agg.merge(std_agg, on=['Season','TeamID'], how='left')
    del g, std_agg; gc.collect()

    eps = 1e-6
    agg['WinPct']     = agg['Win']     / agg['Games']
    agg['AvgScore']   = agg['Score']   / agg['Games']
    agg['AvgAllow']   = agg['Allowed'] / agg['Games']
    agg['AvgMargin']  = agg['AvgScore'] - agg['AvgAllow']
    agg['eFGPct']     = (agg['FGM'] + 0.5*agg['FGM3']) / (agg['FGA'] + eps)
    agg['TOVPct']     = agg['TO'] / (agg['FGA'] + 0.44*agg['FTA'] + agg['TO'] + eps)
    agg['ORBPct']     = agg['OR'] / (agg['OR'] + agg['OppDR'] + eps)
    agg['FTRate']     = agg['FTM'] / (agg['FGA'] + eps)
    agg['FGPct']      = agg['FGM']  / (agg['FGA']  + eps)
    agg['FG3Pct']     = agg['FGM3'] / (agg['FGA3'] + eps)
    agg['AstTORat']   = agg['Ast']  / (agg['TO']   + eps)
    agg['OffPoss']    = (agg['FGA'] - agg['OR'] + agg['TO'] + 0.44*agg['FTA']) / agg['Games']
    agg['OffEffic']   = agg['AvgScore'] / (agg['OffPoss'] + eps)
    agg['DefEffic']   = agg['AvgAllow'] / (agg['OffPoss'] + eps)
    agg['NetEffic']   = agg['OffEffic'] - agg['DefEffic']
    if 'Stl' in agg.columns:
        agg['StlRate'] = agg['Stl'] / (agg['Games'] + eps)
    else:
        agg['StlRate'] = 0.0
    if 'Blk' in agg.columns:
        agg['BlkRate'] = agg['Blk'] / (agg['Games'] + eps)
    else:
        agg['BlkRate'] = 0.0

    keep = ['Season','TeamID','Games','WinPct','AvgScore','AvgAllow','AvgMargin',
            'MarginStd','FTRateStd','PossStd',
            'eFGPct','TOVPct','ORBPct','FTRate','FGPct','FG3Pct','AstTORat',
            'OffPoss','OffEffic','DefEffic','NetEffic','StlRate','BlkRate']
    agg = agg[keep]
    print(f'   Four Factors: {len(agg):,} team-seasons | {agg.shape[1]-2} features')
    return agg

print("   Loading Men's detailed ...")
m_ff = compute_four_factors(M_REG_D_PATH);  gc.collect()
print("   Loading Women's detailed ...")
w_ff = compute_four_factors(W_REG_D_PATH);  gc.collect()

m_ff_lu = m_ff.set_index(['Season','TeamID']).to_dict('index')
w_ff_lu = w_ff.set_index(['Season','TeamID']).to_dict('index')
FF_COLS = [c for c in m_ff.columns if c not in ('Season','TeamID')]



[4/10] Computing Four Factors & box-score features ...
   Loading Men's detailed ...
   Four Factors: 8,346 team-seasons | 21 features
   Loading Women's detailed ...
   Four Factors: 5,965 team-seasons | 21 features


In [6]:
print(f'\n[5/10] Building auxiliary features ...')

def compute_sos(compact_df, elo_lu):
    df     = compact_df[compact_df['DayNum'] <= 132].copy()
    w_side = df[['Season','WTeamID','LTeamID']].rename(columns={'WTeamID':'TeamID','LTeamID':'OppID'})
    l_side = df[['Season','LTeamID','WTeamID']].rename(columns={'LTeamID':'TeamID','WTeamID':'OppID'})
    both   = pd.concat([w_side, l_side], ignore_index=True)
    both['_key']   = list(zip(both['Season'], both['OppID']))
    both['OppElo'] = both['_key'].map(elo_lu).fillna(CFG['elo_init'])
    return both.groupby(['Season','TeamID'])['OppElo'].mean().to_dict()

def compute_sos_winpct(compact_df):
    df     = compact_df[compact_df['DayNum'] <= 132].copy()
    wp     = pd.concat([
        df[['Season','WTeamID']].rename(columns={'WTeamID':'TeamID'}).assign(Win=1),
        df[['Season','LTeamID']].rename(columns={'LTeamID':'TeamID'}).assign(Win=0),
    ]).groupby(['Season','TeamID'])['Win'].mean()
    w_side = df[['Season','WTeamID','LTeamID']].rename(columns={'WTeamID':'TeamID','LTeamID':'OppID'})
    l_side = df[['Season','LTeamID','WTeamID']].rename(columns={'LTeamID':'TeamID','WTeamID':'OppID'})
    both   = pd.concat([w_side, l_side], ignore_index=True)
    both['OppWP'] = both.set_index(['Season','OppID']).index.map(wp.to_dict()).values
    both['OppWP'] = both['OppWP'].fillna(0.5)
    sos_wp = both.groupby(['Season','TeamID'])['OppWP'].mean()
    return sos_wp.to_dict()

def compute_momentum(compact_df):
    df     = compact_df[compact_df['DayNum'] <= 132].copy()
    w_side = df[['Season','DayNum','WTeamID']].rename(columns={'WTeamID':'TeamID'}).assign(Win=1)
    l_side = df[['Season','DayNum','LTeamID']].rename(columns={'LTeamID':'TeamID'}).assign(Win=0)
    tall   = pd.concat([w_side, l_side], ignore_index=True)
    tall.sort_values(['Season','TeamID','DayNum'], inplace=True)
    tall.reset_index(drop=True, inplace=True)
    tall['RevIdx'] = tall.groupby(['Season','TeamID']).cumcount(ascending=False)
    last10 = tall[tall['RevIdx'] < 10].copy()
    last10['Weight'] = 0.85 ** last10['RevIdx']
    grp   = last10.groupby(['Season','TeamID'])
    wsum  = grp.apply(lambda x: (x['Win'] * x['Weight']).sum())
    dsum  = grp['Weight'].sum()
    mom   = (wsum / dsum).reset_index()
    mom.columns = ['Season','TeamID','Momentum']
    del tall, last10, w_side, l_side; gc.collect()
    return mom.set_index(['Season','TeamID'])['Momentum'].to_dict()

def compute_momentum_last5(compact_df):
    df     = compact_df[compact_df['DayNum'] <= 132].copy()
    w_side = df[['Season','DayNum','WTeamID']].rename(columns={'WTeamID':'TeamID'}).assign(Win=1)
    l_side = df[['Season','DayNum','LTeamID']].rename(columns={'LTeamID':'TeamID'}).assign(Win=0)
    tall   = pd.concat([w_side, l_side], ignore_index=True)
    tall.sort_values(['Season','TeamID','DayNum'], inplace=True)
    tall['RevIdx'] = tall.groupby(['Season','TeamID']).cumcount(ascending=False)
    last5  = tall[tall['RevIdx'] < 5]
    mom5   = last5.groupby(['Season','TeamID'])['Win'].mean().reset_index()
    mom5.columns = ['Season','TeamID','Mom5']
    del tall, last5, w_side, l_side; gc.collect()
    return mom5.set_index(['Season','TeamID'])['Mom5'].to_dict()

def compute_conf_features(compact_df, conf_df):
    conf_lu    = conf_df.set_index(['Season','TeamID'])['ConfAbbrev'].to_dict()
    w_conf_mg  = compact_df.merge(
        conf_df.rename(columns={'TeamID':'WTeamID','ConfAbbrev':'WConf'}),
        on=['Season','WTeamID'], how='left')
    l_conf_mg  = compact_df.merge(
        conf_df.rename(columns={'TeamID':'LTeamID','ConfAbbrev':'LConf'}),
        on=['Season','LTeamID'], how='left')
    ww = w_conf_mg.groupby(['Season','WConf']).size().reset_index(name='W').rename(columns={'WConf':'Conf'})
    ll = l_conf_mg.groupby(['Season','LConf']).size().reset_index(name='L').rename(columns={'LConf':'Conf'})
    cdf = ww.merge(ll, on=['Season','Conf'], how='outer').fillna(0)
    cdf['WinRate'] = cdf['W'] / (cdf['W'] + cdf['L'] + 1e-6)
    conf_wr_lu = cdf.set_index(['Season','Conf'])['WinRate'].to_dict()
    return conf_lu, conf_wr_lu

def build_seed_lu(seeds_df):
    df = seeds_df.copy()
    df['SeedNum'] = df['Seed'].str.extract(r'(\d+)').astype(int)
    return df.set_index(['Season','TeamID'])['SeedNum'].to_dict()

def compute_coach_tourney_records(coaches_df, tourney_df):
    active    = coaches_df[coaches_df['LastDayNum'] >= 132].copy()
    coach_map = active[['Season','TeamID','CoachName']].drop_duplicates(subset=['Season','TeamID'])
    tw = tourney_df.merge(
        coach_map.rename(columns={'TeamID':'WTeamID','CoachName':'WCoach'}),
        on=['Season','WTeamID'], how='left')
    tl = tourney_df.merge(
        coach_map.rename(columns={'TeamID':'LTeamID','CoachName':'LCoach'}),
        on=['Season','LTeamID'], how='left')
    coach_wins   = tw.groupby('WCoach').size().reset_index(name='TWins').rename(columns={'WCoach':'Coach'})
    coach_losses = tl.groupby('LCoach').size().reset_index(name='TLosses').rename(columns={'LCoach':'Coach'})
    crec = coach_wins.merge(coach_losses, on='Coach', how='outer').fillna(0)
    crec['CoachTWR']   = crec['TWins'] / (crec['TWins'] + crec['TLosses'] + 1e-6)
    crec['CoachGames'] = crec['TWins'] + crec['TLosses']
    coach_twr   = crec.set_index('Coach')['CoachTWR'].to_dict()
    coach_games = crec.set_index('Coach')['CoachGames'].to_dict()
    coach_name_map = coach_map.set_index(['Season','TeamID'])['CoachName'].to_dict()
    result = {}
    for (s, tid), cname in coach_name_map.items():
        result[(s, tid)] = (coach_twr.get(cname, 0.5), coach_games.get(cname, 0))
    return result

def compute_h2h(m_tourney, w_tourney, prior_n=3):
    all_t = pd.concat([m_tourney, w_tourney], ignore_index=True)[['Season','WTeamID','LTeamID']].copy()
    all_t['T1']    = all_t[['WTeamID','LTeamID']].min(axis=1)
    all_t['T2']    = all_t[['WTeamID','LTeamID']].max(axis=1)
    all_t['T1Win'] = (all_t['WTeamID'] == all_t['T1']).astype(int)
    grp = all_t.groupby(['T1','T2'])['T1Win'].agg(['sum','count'])
    grp.columns = ['Wins','Games']
    grp['H2H']  = (grp['Wins'] + prior_n * 0.5) / (grp['Games'] + prior_n)
    win_lu   = grp['H2H'].to_dict()
    games_lu = grp['Games'].clip(upper=6).to_dict()
    print(f'   H2H: {len(win_lu):,} unique matchup pairs')
    return win_lu, games_lu

def compute_tourney_experience(m_tourney, w_tourney, m_seeds, w_seeds, window=6):
    all_t    = pd.concat([m_tourney[['Season','WTeamID','LTeamID']],
                          w_tourney[['Season','WTeamID','LTeamID']]], ignore_index=True)
    wins     = all_t.groupby(['Season','WTeamID']).size().reset_index()
    wins.columns = ['Season','TeamID','Wins']
    all_seeds = pd.concat([m_seeds[['Season','TeamID','Seed']],
                           w_seeds[['Season','TeamID','Seed']]], ignore_index=True)
    all_seeds['SeedNum'] = all_seeds['Seed'].str[1:3].astype(int)
    apps = all_seeds.merge(wins, on=['Season','TeamID'], how='left').fillna({'Wins': 0})
    apps['Wins']    = apps['Wins'].astype(int)
    apps['DeepRun'] = (apps['Wins'] >= 2).astype(int)
    apps['EliteEight'] = (apps['Wins'] >= 3).astype(int)
    lu = {}
    for season in sorted(apps['Season'].unique()):
        past = apps[apps['Season'].between(season - window, season - 1)]
        for team_id, grp in past.groupby('TeamID'):
            n_apps = len(grp)
            lu[(season, int(team_id))] = {
                'TourneyApps'  : n_apps,
                'TourneyWPG'   : grp['Wins'].mean() if n_apps else 0.0,
                'DeepRunRate'  : grp['DeepRun'].mean() if n_apps else 0.0,
                'EliteEightRate': grp['EliteEight'].mean() if n_apps else 0.0,
                'SeedAvg'      : grp['SeedNum'].mean() if n_apps else 8.5,
                'SeedBest'     : grp['SeedNum'].min() if n_apps else 16,
            }
    n_teams = len({k[1] for k in lu})
    print(f'   TourneyExp: {len(lu):,} (season, team) records | {n_teams:,} teams')
    return lu

m_sos_lu    = compute_sos(m_reg_c, m_elo_lu)
w_sos_lu    = compute_sos(w_reg_c, w_elo_lu)
m_soswp_lu  = compute_sos_winpct(m_reg_c)
w_soswp_lu  = compute_sos_winpct(w_reg_c)
print(f'   SOS: Men {len(m_sos_lu):,} | Women {len(w_sos_lu):,}')

m_mom_lu    = compute_momentum(m_reg_c)
w_mom_lu    = compute_momentum(w_reg_c)
m_mom5_lu   = compute_momentum_last5(m_reg_c)
w_mom5_lu   = compute_momentum_last5(w_reg_c)
print(f'   Momentum: Men {len(m_mom_lu):,} | Women {len(w_mom_lu):,}')

m_conf_lu, m_conf_wr_lu = compute_conf_features(m_reg_c, m_conf)
w_conf_lu, w_conf_wr_lu = compute_conf_features(w_reg_c, w_conf)
print(f'   Conference: Men {len(m_conf_wr_lu):,} | Women {len(w_conf_wr_lu):,}')

m_seed_lu   = build_seed_lu(m_seeds)
w_seed_lu   = build_seed_lu(w_seeds)

m_coach_lu  = compute_coach_tourney_records(coaches, m_tour_c)
print(f'   Coach records: {len(m_coach_lu):,} team-seasons')

h2h_lu, h2h_games_lu = compute_h2h(m_tour_c, w_tour_c)
tourney_exp_lu        = compute_tourney_experience(m_tour_c, w_tour_c, m_seeds, w_seeds)



[5/10] Building auxiliary features ...
   SOS: Men 13,753 | Women 9,851
   Momentum: Men 13,753 | Women 9,851
   Conference: Men 1,360 | Women 930
   Coach records: 13,398 team-seasons
   H2H: 3,650 unique matchup pairs
   TourneyExp: 9,754 (season, team) records | 588 teams


In [7]:
print(f'\n[6/10] Assembling feature matrix ...')

def get_team_features(season, team_id, elo_lu, ff_lu, sos_lu, soswp_lu,
                      mom_lu, mom5_lu, seed_lu, conf_lu, conf_wr_lu,
                      off_lu, def_lu, mas_lu=None, coach_lu=None):
    init = CFG['elo_init']
    elo   = elo_lu.get((season, team_id), init)
    ff    = ff_lu.get((season, team_id), {})
    sos   = sos_lu.get((season, team_id), init)
    soswp = soswp_lu.get((season, team_id), 0.5)
    mom   = mom_lu.get((season, team_id), 0.5)
    mom5  = mom5_lu.get((season, team_id), 0.5)
    seed  = seed_lu.get((season, team_id), 8.5)
    conf  = conf_lu.get((season, team_id), 'unk')
    conf_wr = conf_wr_lu.get((season, conf), 0.5)
    is_pwr  = 1 if conf in CFG['power_confs'] else 0
    off_rtg = off_lu.get((season, team_id), init)
    def_rtg = def_lu.get((season, team_id), init)

    if coach_lu:
        ctwr, cgames = coach_lu.get((season, team_id), (0.5, 0))
    else:
        ctwr, cgames = 0.5, 0

    avg_sc  = ff.get('AvgScore', 70.0)
    avg_all = ff.get('AvgAllow', 70.0)
    _e      = 1e-6
    pyth    = avg_sc**11.5 / (avg_sc**11.5 + avg_all**11.5 + _e)

    win_pct = ff.get('WinPct', 0.5)
    avg_mgn = ff.get('AvgMargin', 0.0)
    net_eff = ff.get('NetEffic', 0.0)
    off_eff = ff.get('OffEffic', 1.0)
    def_eff = ff.get('DefEffic', 1.0)

    exp   = tourney_exp_lu.get((season, team_id), {})

    feats = {
        'Elo'           : elo,
        'OffRtg'        : off_rtg,
        'DefRtg'        : def_rtg,
        'NetRtg'        : off_rtg - def_rtg,
        'WinPct'        : win_pct,
        'AvgScore'      : avg_sc,
        'AvgAllowed'    : avg_all,
        'AvgMargin'     : avg_mgn,
        'MarginStd'     : ff.get('MarginStd',  8.0),
        'eFGPct'        : ff.get('eFGPct',   0.490),
        'TOVPct'        : ff.get('TOVPct',   0.170),
        'ORBPct'        : ff.get('ORBPct',   0.300),
        'FTRate'        : ff.get('FTRate',   0.240),
        'FTRateStd'     : ff.get('FTRateStd',0.060),
        'FGPct'         : ff.get('FGPct',    0.440),
        'FG3Pct'        : ff.get('FG3Pct',   0.330),
        'AstTORat'      : ff.get('AstTORat', 1.100),
        'OffEffic'      : off_eff,
        'DefEffic'      : def_eff,
        'NetEffic'      : net_eff,
        'StlRate'       : ff.get('StlRate',  0.060),
        'BlkRate'       : ff.get('BlkRate',  0.040),
        'PossStd'       : ff.get('PossStd',  2.0),
        'Pyth'          : pyth,
        'SOS'           : sos,
        'SOSWinPct'     : soswp,
        'Momentum'      : mom,
        'Mom5'          : mom5,
        'Seed'          : seed,
        'ConfWR'        : conf_wr,
        'IsPower'       : is_pwr,
        'CoachTWR'      : ctwr,
        'CoachGames'    : cgames,
        'TourneyApps'   : exp.get('TourneyApps',    0.0),
        'TourneyWPG'    : exp.get('TourneyWPG',     0.0),
        'DeepRunRate'   : exp.get('DeepRunRate',    0.0),
        'EliteEightRate': exp.get('EliteEightRate', 0.0),
        'SeedAvg'       : exp.get('SeedAvg',        8.5),
        'SeedBest'      : exp.get('SeedBest',       16),
    }

    if mas_lu is not None:
        m = mas_lu.get((season, team_id), {})
        feats['MasMean']   = m.get('mas_mean',    0.5)
        feats['MasMin']    = m.get('mas_min',     0.5)
        feats['MasMax']    = m.get('mas_max',     0.5)
        feats['MasStd']    = m.get('mas_std',     0.1)
        feats['MasTop3']   = m.get('mas_top3_mean',0.5)
        feats['MasPOM']    = m.get('mas_POM',     0.5)
        feats['MasSAG']    = m.get('mas_SAG',     0.5)
    else:
        for k in ['MasMean','MasMin','MasMax','MasStd','MasTop3','MasPOM','MasSAG']:
            feats[k] = 0.5

    return feats


TEAM_FEAT_NAMES = list(get_team_features(
    2020, 1234, m_elo_lu, m_ff_lu, m_sos_lu, m_soswp_lu,
    m_mom_lu, m_mom5_lu, m_seed_lu, m_conf_lu, m_conf_wr_lu,
    m_off_lu, m_def_lu, massey_lu, m_coach_lu
).keys())


def build_matchup_features(season, t1_id, t2_id, is_women):
    if is_women:
        elo_lu  = w_elo_lu;  ff_lu   = w_ff_lu
        sos_lu  = w_sos_lu;  soswp_lu= w_soswp_lu
        mom_lu  = w_mom_lu;  mom5_lu = w_mom5_lu
        seed_lu = w_seed_lu; conf_lu = w_conf_lu; conf_wr = w_conf_wr_lu
        off_lu  = w_off_lu;  def_lu  = w_def_lu
        mas_lu  = None;      coach_lu= None
    else:
        elo_lu  = m_elo_lu;  ff_lu   = m_ff_lu
        sos_lu  = m_sos_lu;  soswp_lu= m_soswp_lu
        mom_lu  = m_mom_lu;  mom5_lu = m_mom5_lu
        seed_lu = m_seed_lu; conf_lu = m_conf_lu; conf_wr = m_conf_wr_lu
        off_lu  = m_off_lu;  def_lu  = m_def_lu
        mas_lu  = massey_lu; coach_lu= m_coach_lu

    f1 = get_team_features(season, t1_id, elo_lu, ff_lu, sos_lu, soswp_lu,
                           mom_lu, mom5_lu, seed_lu, conf_lu, conf_wr,
                           off_lu, def_lu, mas_lu, coach_lu)
    f2 = get_team_features(season, t2_id, elo_lu, ff_lu, sos_lu, soswp_lu,
                           mom_lu, mom5_lu, seed_lu, conf_lu, conf_wr,
                           off_lu, def_lu, mas_lu, coach_lu)

    row = {}
    for k in TEAM_FEAT_NAMES:
        row[f'T1_{k}'] = f1[k]
        row[f'T2_{k}'] = f2[k]
        row[f'D_{k}']  = f1[k] - f2[k]

    h2h_key = (t1_id, t2_id)
    row['H2H_WinRate']  = h2h_lu.get(h2h_key, 0.5)
    row['H2H_Games']    = h2h_games_lu.get(h2h_key, 0)

    elo_diff  = row['D_Elo']
    seed_diff = row['D_Seed']
    net_diff  = row['D_AvgMargin']
    pyth_diff = row['D_Pyth']
    eff_diff  = row['D_NetEffic']
    rtg_diff  = row['D_NetRtg']

    row['IX_Elo_x_Seed']    = elo_diff  * seed_diff
    row['IX_Net_x_Seed']    = net_diff  * seed_diff
    row['IX_Elo_x_Net']     = elo_diff  * net_diff
    row['IX_Seed_x_Pyth']   = seed_diff * pyth_diff
    row['IX_Off_x_Def']     = row['D_eFGPct'] * (-row['D_TOVPct'])
    row['IX_Eff_x_Seed']    = eff_diff  * seed_diff
    row['IX_Rtg_x_Seed']    = rtg_diff  * seed_diff
    row['IX_Mom_x_Elo']     = row['D_Momentum'] * elo_diff
    row['IX_Mom5_x_Elo']    = row['D_Mom5'] * elo_diff
    row['IX_SOS_x_WinPct']  = row['D_SOS'] * row['D_WinPct']
    row['EloWinProb']       = 1.0 / (1.0 + 10 ** (-elo_diff / 400.0))
    row['SeedWinProb']      = 1.0 / (1.0 + np.exp(0.4 * seed_diff))
    row['RtgWinProb']       = 1.0 / (1.0 + np.exp(-rtg_diff / 80.0))
    row['PythWinProb']      = f1['Pyth'] / (f1['Pyth'] + f2['Pyth'] + 1e-9)
    return row


sample_row = build_matchup_features(2024, 1101, 1200, is_women=False)
N_FEATS    = len(sample_row)
FEAT_COLS  = list(sample_row.keys())
print(f'   Feature vector: {N_FEATS} features per matchup')
print(f'   ({len(TEAM_FEAT_NAMES)} team features × 3 views + interactions)')



[6/10] Assembling feature matrix ...
   Feature vector: 154 features per matchup
   (46 team features × 3 views + interactions)


In [8]:
print(f'\n[7/10] Building training dataset ...')

def build_training_set(tourney_df, is_women, min_season=CFG['min_season']):
    df   = tourney_df[tourney_df['Season'] >= min_season].copy()
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df),
                     desc=f"  {'Women' if is_women else 'Men  ':5s} tourney rows"):
        s, wid, lid = r['Season'], r['WTeamID'], r['LTeamID']
        if wid < lid:
            t1, t2, outcome = wid, lid, 1
        else:
            t1, t2, outcome = lid, wid, 0
        feats            = build_matchup_features(s, t1, t2, is_women)
        feats['Outcome'] = outcome
        feats['Season']  = s
        rows.append(feats)
    result = pd.DataFrame(rows)
    del rows; gc.collect()
    return result

m_train = build_training_set(m_tour_c, is_women=False)
w_train = build_training_set(w_tour_c, is_women=True)

m_train['IsWomen'] = 0
w_train['IsWomen'] = 1
train_base = pd.concat([m_train, w_train], ignore_index=True)

t1_cols = [c for c in train_base.columns if c.startswith('T1_')]
t2_cols = [c for c in train_base.columns if c.startswith('T2_')]
d_cols  = [c for c in train_base.columns if c.startswith('D_')]

train_aug          = train_base.copy()
train_aug[t1_cols] = train_base[t2_cols].values
train_aug[t2_cols] = train_base[t1_cols].values
train_aug[d_cols]  = -train_base[d_cols].values
train_aug['Outcome'] = 1 - train_base['Outcome']

train_df = pd.concat([train_base, train_aug], ignore_index=True)
del train_base, train_aug; gc.collect()

print(f'\n   Base rows        : {len(m_train) + len(w_train):,}')
print(f'   After augmentation: {len(train_df):,}')
print(f'   Class balance     : {train_df["Outcome"].mean():.3f}')



[7/10] Building training dataset ...


  Women tourney rows: 100%|██████████| 1213/1213 [00:00<00:00, 8878.47it/s]



   Base rows        : 2,470
   After augmentation: 4,940
   Class balance     : 0.500


In [9]:
print(f'\n[8/10] Training models ...')

import torch
USE_GPU    = torch.cuda.is_available()
GPU_NAME   = torch.cuda.get_device_name(0) if USE_GPU else 'None'
LGB_DEVICE = 'gpu'      if USE_GPU else 'cpu'
XGB_TREE   = 'hist'
CAT_TASK   = 'GPU'      if USE_GPU else 'CPU'
TORCH_DEV  = torch.device('cuda' if USE_GPU else 'cpu')
print(f'   GPU : {USE_GPU}  ({GPU_NAME})')


def shadow_feature_selection(train_df, feat_cols, gender, val_seasons,
                             n_top=70, n_runs=7, threshold=0.50, held_out=5):
    safe_df  = train_df[~train_df['Season'].isin(val_seasons)].copy()
    unique_s = sorted(safe_df['Season'].unique())
    safe_df  = safe_df[safe_df['Season'].isin(unique_s[:-held_out])]
    if len(safe_df) < 100:
        return feat_cols

    X_safe      = safe_df[feat_cols].values.astype(np.float32)
    y_safe      = safe_df['Outcome'].values
    col_medians = np.nanmedian(X_safe, axis=0)
    nan_mask    = np.isnan(X_safe)
    X_safe[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])

    print(f'   {gender} feature selection on {len(safe_df):,} games ...')

    probe = lgb.LGBMClassifier(n_estimators=400, num_leaves=31, max_depth=5,
                                learning_rate=0.04, random_state=CFG['seed'], verbose=-1,
                                device=LGB_DEVICE)
    probe.fit(X_safe, y_safe)
    imp       = pd.Series(probe.feature_importances_, index=feat_cols)
    top_feats = imp.nlargest(n_top).index.tolist()
    top_idx   = [feat_cols.index(f) for f in top_feats]
    X_top     = X_safe[:, top_idx]

    shadow_wins = np.zeros(len(top_feats))
    for run in range(n_runs):
        rng      = np.random.RandomState(CFG['seed'] + run)
        X_shadow = np.array([rng.permutation(X_top[:, j]) for j in range(X_top.shape[1])]).T
        X_aug    = np.hstack([X_top, X_shadow])
        probe2   = lgb.LGBMClassifier(n_estimators=250, num_leaves=15, max_depth=4,
                                       learning_rate=0.04, random_state=CFG['seed']+run,
                                       verbose=-1, device=LGB_DEVICE)
        probe2.fit(X_aug, y_safe)
        imp2        = probe2.feature_importances_
        shadow_wins += (imp2[:len(top_feats)] > imp2[len(top_feats):].max()).astype(float)

    keep_mask = (shadow_wins / n_runs) >= threshold
    selected  = [f for f, k in zip(top_feats, keep_mask) if k]

    must_keep = [
        'D_Elo','D_Seed','D_MasMean','D_MasPOM','D_WinPct','D_AvgMargin',
        'D_eFGPct','D_Pyth','D_NetEffic','D_NetRtg','D_Momentum','D_Mom5',
        'H2H_WinRate','EloWinProb','SeedWinProb','RtgWinProb','PythWinProb',
        'D_TourneyApps','D_DeepRunRate','D_EliteEightRate',
        'D_OffEffic','D_DefEffic','D_MasTop3',
    ]
    for f in must_keep:
        if f in feat_cols and f not in selected:
            selected.append(f)

    seen = set()
    selected = [f for f in selected if not (f in seen or seen.add(f))]
    print(f'   {gender} selected {len(selected)} / {len(feat_cols)} features')
    return selected


def train_gender_model(train_df_g, val_df_g, feat_cols, gender):
    X_tr  = train_df_g[feat_cols]
    y_tr  = train_df_g['Outcome']
    X_val = val_df_g[feat_cols]
    y_val = val_df_g['Outcome']

    decay      = CFG['recency_decay']
    max_season = int(train_df_g['Season'].max())
    w_tr       = decay ** (max_season - train_df_g['Season'].values)
    w_tr       = w_tr / w_tr.mean()

    all_df    = pd.concat([train_df_g, val_df_g], ignore_index=True)
    X_all     = all_df[feat_cols]
    y_all     = all_df['Outcome']
    max_s_all = int(all_df['Season'].max())
    w_all     = decay ** (max_s_all - all_df['Season'].values)
    w_all     = w_all / w_all.mean()

    print(f'\n   {gender}: train={len(X_tr):,}  val={len(X_val):,}')
    print(f'   Tuning {gender} hyperparameters (Optuna TPE) ...')

    X_np = X_tr.values if hasattr(X_tr, 'values') else np.array(X_tr)
    y_np = y_tr.values if hasattr(y_tr, 'values') else np.array(y_tr)
    s_np = train_df_g['Season'].values
    w_np = w_tr
    _state = [None]

    def _temporal_brier(model_fn, X, y, seasons, weights):
        unique_s = sorted(np.unique(seasons))
        if len(unique_s) < 4:
            return 0.25
        preds, trues = [], []
        for ts in unique_s[-7:]:
            tr_idx = np.where(seasons < ts)[0]
            te_idx = np.where(seasons == ts)[0]
            if len(tr_idx) < 40 or len(te_idx) == 0:
                continue
            model_fn(X[tr_idx], y[tr_idx], weights[tr_idx], X[te_idx])
            preds.append(_state[0])
            trues.append(y[te_idx])
        if not preds:
            return 0.25
        return brier_score_loss(np.concatenate(trues), np.concatenate(preds))

    def _lgb_objective(trial):
        p = {
            'objective'        : 'binary',
            'metric'           : 'binary_logloss',
            'verbose'          : -1,
            'random_state'     : CFG['seed'],
            'device'           : LGB_DEVICE,
            'gpu_use_dp'       : True,
            'learning_rate'    : trial.suggest_float('lr',  0.005, 0.08, log=True),
            'num_leaves'       : trial.suggest_int('nl',    16, 128),
            'max_depth'        : trial.suggest_int('md',    3, 8),
            'min_child_samples': trial.suggest_int('mcs',   3, 40),
            'feature_fraction' : trial.suggest_float('ff',  0.4, 1.0),
            'bagging_fraction' : trial.suggest_float('bf',  0.4, 1.0),
            'bagging_freq'     : 5,
            'reg_alpha'        : trial.suggest_float('ra',  1e-3, 8.0, log=True),
            'reg_lambda'       : trial.suggest_float('rl',  1e-3, 12.0, log=True),
            'min_split_gain'   : trial.suggest_float('msg', 0.0, 1.0),
        }
        def _fit(Xtr, ytr, wtr, Xte):
            ds = lgb.Dataset(Xtr, label=ytr, weight=wtr)
            m  = lgb.train(p, ds, num_boost_round=400, callbacks=[lgb.log_evaluation(-1)])
            _state[0] = m.predict(Xte)
        return _temporal_brier(_fit, X_np, y_np, s_np, w_np)

    lgb_study = optuna.create_study(direction='minimize',
                                    sampler=TPESampler(seed=CFG['seed'], n_startup_trials=10, multivariate=True))
    lgb_study.optimize(_lgb_objective, n_trials=CFG['lgb_n_trials'], show_progress_bar=False)
    lgb_p_opt = lgb_study.best_params
    lgb_p = {
        'objective': 'binary', 'metric': 'binary_logloss', 'verbose': -1,
        'random_state': CFG['seed'], 'bagging_freq': 5,
        'device': LGB_DEVICE, 'gpu_use_dp': True,
        'learning_rate': lgb_p_opt['lr'], 'num_leaves': lgb_p_opt['nl'],
        'max_depth': lgb_p_opt['md'], 'min_child_samples': lgb_p_opt['mcs'],
        'feature_fraction': lgb_p_opt['ff'], 'bagging_fraction': lgb_p_opt['bf'],
        'reg_alpha': lgb_p_opt['ra'], 'reg_lambda': lgb_p_opt['rl'],
        'min_split_gain': lgb_p_opt['msg'],
    }
    print(f'   {gender} LGB best CV Brier={lgb_study.best_value:.5f}')

    def _xgb_objective(trial):
        p = {
            'objective'       : 'binary:logistic',
            'eval_metric'     : 'logloss',
            'verbosity'       : 0,
            'seed'            : CFG['seed'],
            'tree_method'     : XGB_TREE,
            'device'          : 'cuda' if USE_GPU else 'cpu',
            'learning_rate'   : trial.suggest_float('lr',  0.005, 0.08, log=True),
            'max_depth'       : trial.suggest_int('md',    3, 8),
            'subsample'       : trial.suggest_float('ss',  0.4, 1.0),
            'colsample_bytree': trial.suggest_float('cb',  0.4, 1.0),
            'min_child_weight': trial.suggest_int('mcw',   1, 20),
            'gamma'           : trial.suggest_float('g',   0.0, 4.0),
            'reg_alpha'       : trial.suggest_float('ra',  1e-3, 8.0, log=True),
            'reg_lambda'      : trial.suggest_float('rl',  1e-3, 12.0, log=True),
            'max_delta_step'  : trial.suggest_int('mds',   0, 3),
        }
        def _fit(Xtr, ytr, wtr, Xte):
            dm  = xgb.DMatrix(Xtr, label=ytr, weight=wtr)
            m   = xgb.train(p, dm, num_boost_round=400)
            _state[0] = m.predict(xgb.DMatrix(Xte))
        return _temporal_brier(_fit, X_np, y_np, s_np, w_np)

    xgb_study = optuna.create_study(direction='minimize',
                                    sampler=TPESampler(seed=CFG['seed']+1, n_startup_trials=8, multivariate=True))
    xgb_study.optimize(_xgb_objective, n_trials=CFG['xgb_n_trials'], show_progress_bar=False)
    xgb_p_opt = xgb_study.best_params
    xgb_p = {
        'objective': 'binary:logistic', 'eval_metric': 'logloss',
        'verbosity': 0, 'seed': CFG['seed'],
        'tree_method': XGB_TREE, 'device': 'cuda' if USE_GPU else 'cpu',
        'learning_rate': xgb_p_opt['lr'], 'max_depth': xgb_p_opt['md'],
        'subsample': xgb_p_opt['ss'], 'colsample_bytree': xgb_p_opt['cb'],
        'min_child_weight': xgb_p_opt['mcw'], 'gamma': xgb_p_opt['g'],
        'reg_alpha': xgb_p_opt['ra'], 'reg_lambda': xgb_p_opt['rl'],
        'max_delta_step': xgb_p_opt['mds'],
    }
    print(f'   {gender} XGB best CV Brier={xgb_study.best_value:.5f}')

    def _cat_objective(trial):
        p = {
            'verbose': 0, 'random_seed': CFG['seed'], 'iterations': 400,
            'task_type'          : CAT_TASK,
            'learning_rate'      : trial.suggest_float('lr',  0.005, 0.08, log=True),
            'depth'              : trial.suggest_int('d',     3, 8),
            'l2_leaf_reg'        : trial.suggest_float('l2',  1.0, 25.0, log=True),
            'bagging_temperature': trial.suggest_float('bt',  0.0, 2.5),
            'random_strength'    : trial.suggest_float('rs',  0.0, 2.5),
            'border_count'       : trial.suggest_int('bc',    32, 255),
        }
        def _fit(Xtr, ytr, wtr, Xte):
            m = CatBoostClassifier(**p)
            m.fit(Xtr, ytr, sample_weight=wtr, verbose=False)
            _state[0] = m.predict_proba(Xte)[:, 1]
        return _temporal_brier(_fit, X_np, y_np, s_np, w_np)

    cat_study = optuna.create_study(direction='minimize',
                                    sampler=TPESampler(seed=CFG['seed']+2, n_startup_trials=10, multivariate=True))
    cat_study.optimize(_cat_objective, n_trials=CFG['cat_n_trials'], show_progress_bar=False)
    cat_p_opt = cat_study.best_params
    cat_p = {
        'verbose': 0, 'random_seed': CFG['seed'], 'iterations': 800,
        'task_type': CAT_TASK,
        'learning_rate': cat_p_opt['lr'], 'depth': cat_p_opt['d'],
        'l2_leaf_reg': cat_p_opt['l2'], 'bagging_temperature': cat_p_opt['bt'],
        'random_strength': cat_p_opt['rs'], 'border_count': cat_p_opt['bc'],
    }
    print(f'   {gender} CAT best CV Brier={cat_study.best_value:.5f}')

    lgb_tr_ds  = lgb.Dataset(X_tr,  label=y_tr,  weight=w_tr)
    lgb_val_ds = lgb.Dataset(X_val, label=y_val)

    lgb_cv = lgb.train(lgb_p, lgb_tr_ds, num_boost_round=4000,
                       valid_sets=[lgb_val_ds],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                   lgb.log_evaluation(200)])
    lgb_val_p = lgb_cv.predict(X_val)
    lgb_b     = brier_score_loss(y_val, lgb_val_p)
    print(f'   {gender} LightGBM  Brier={lgb_b:.5f}  iter={lgb_cv.best_iteration}')
    lgb_full  = lgb.train(lgb_p, lgb.Dataset(X_all, label=y_all, weight=w_all),
                          num_boost_round=lgb_cv.best_iteration)

    dtr  = xgb.DMatrix(X_tr,  label=y_tr,  weight=w_tr)
    dvl  = xgb.DMatrix(X_val, label=y_val)
    dall = xgb.DMatrix(X_all, label=y_all, weight=w_all)
    xgb_cv = xgb.train(xgb_p, dtr, num_boost_round=4000,
                        evals=[(dvl,'val')], early_stopping_rounds=100,
                        verbose_eval=False)
    xgb_val_p = xgb_cv.predict(dvl)
    xgb_b     = brier_score_loss(y_val, xgb_val_p)
    print(f'   {gender} XGBoost   Brier={xgb_b:.5f}  iter={xgb_cv.best_iteration}')
    xgb_full  = xgb.train(xgb_p, dall, num_boost_round=xgb_cv.best_iteration)

    cat_cv = CatBoostClassifier(**cat_p)
    cat_cv.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=(X_val, y_val),
               early_stopping_rounds=60)
    cat_val_p = cat_cv.predict_proba(X_val)[:, 1]
    cat_b     = brier_score_loss(y_val, cat_val_p)
    print(f'   {gender} CatBoost  Brier={cat_b:.5f}  iter={cat_cv.best_iteration_}')
    cat_full = CatBoostClassifier(**cat_p)
    cat_full.fit(X_all, y_all, sample_weight=w_all)

    _imp = SimpleImputer(strategy='median')
    _sca = StandardScaler()
    X_tr_lr  = _sca.fit_transform(_imp.fit_transform(X_tr))
    X_val_lr = _sca.transform(_imp.transform(X_val))
    X_all_lr = _sca.transform(_imp.transform(X_all))

    lr_cv  = LogisticRegression(C=0.3, penalty='l2', solver='lbfgs',
                                 max_iter=3000, random_state=CFG['seed'])
    lr_cv.fit(X_tr_lr, y_tr, sample_weight=w_tr)
    lr_val_p = lr_cv.predict_proba(X_val_lr)[:, 1]
    lr_b     = brier_score_loss(y_val, lr_val_p)
    print(f'   {gender} LogReg    Brier={lr_b:.5f}')
    lr_full = LogisticRegression(C=0.3, penalty='l2', solver='lbfgs',
                                  max_iter=3000, random_state=CFG['seed'])
    lr_full.fit(X_all_lr, y_all, sample_weight=w_all)

    mlp_cv  = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                             max_iter=500, random_state=CFG['seed'],
                             early_stopping=True, validation_fraction=0.15,
                             learning_rate_init=0.001, alpha=0.01)
    mlp_cv.fit(X_tr_lr, y_tr)
    mlp_val_p = mlp_cv.predict_proba(X_val_lr)[:, 1]
    mlp_b     = brier_score_loss(y_val, mlp_val_p)
    print(f'   {gender} MLP       Brier={mlp_b:.5f}')
    mlp_full = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                              max_iter=500, random_state=CFG['seed'],
                              learning_rate_init=0.001, alpha=0.01)
    mlp_full.fit(X_all_lr, y_all)

    inv_l = 1.0 / (lgb_b + 1e-9)
    inv_x = 1.0 / (xgb_b + 1e-9)
    inv_c = 1.0 / (cat_b + 1e-9)
    inv_r = 1.0 / (lr_b  + 1e-9)
    inv_m = 1.0 / (mlp_b + 1e-9)
    tot   = inv_l + inv_x + inv_c + inv_r + inv_m
    wl, wx, wc, wr, wm = inv_l/tot, inv_x/tot, inv_c/tot, inv_r/tot, inv_m/tot

    ens_val_p = wl*lgb_val_p + wx*xgb_val_p + wc*cat_val_p + wr*lr_val_p + wm*mlp_val_p
    ens_b     = brier_score_loss(y_val, ens_val_p)
    print(f'   {gender} Ensemble  Brier={ens_b:.5f}  '
          f'(LGB={wl:.2f} XGB={wx:.2f} CAT={wc:.2f} LR={wr:.2f} MLP={wm:.2f})')

    iso_g    = IsotonicRegression(out_of_bounds='clip')
    iso_g.fit(ens_val_p, y_val)
    cal_val_p = iso_g.predict(ens_val_p)
    cal_b     = brier_score_loss(y_val, cal_val_p)
    print(f'   {gender} Calibrated Brier={cal_b:.5f}  (Δ={cal_b-ens_b:+.5f})')

    return {
        'lgb_model'    : lgb_full,
        'xgb_model'    : xgb_full,
        'cat_model'    : cat_full,
        'lr_model'     : lr_full,
        'mlp_model'    : mlp_full,
        'lr_imp'       : _imp,
        'lr_sca'       : _sca,
        'iso_cal'      : iso_g,
        'w_lgb': wl, 'w_xgb': wx, 'w_cat': wc, 'w_lr': wr, 'w_mlp': wm,
        'lgb_brier': lgb_b, 'xgb_brier': xgb_b, 'cat_brier': cat_b,
        'lr_brier': lr_b, 'mlp_brier': mlp_b,
        'ens_brier': ens_b, 'cal_brier': cal_b,
        'lgb_cv_model' : lgb_cv,
        'xgb_cv_model' : xgb_cv,
        'val_preds_ens': ens_val_p,
        'val_preds_cal': cal_val_p,
        'y_val'        : y_val.values,
        'X_val'        : X_val,
        'feat_cols'    : feat_cols,
    }


val_mask    = train_df['Season'].isin(CFG['val_seasons'])
m_train_cut = train_df[(~val_mask) & (train_df['IsWomen'] == 0)].copy()
m_val_cut   = train_df[( val_mask) & (train_df['IsWomen'] == 0)].copy()
w_train_cut = train_df[(~val_mask) & (train_df['IsWomen'] == 1)].copy()
w_val_cut   = train_df[( val_mask) & (train_df['IsWomen'] == 1)].copy()
m_all_df    = train_df[train_df['IsWomen'] == 0].copy()
w_all_df    = train_df[train_df['IsWomen'] == 1].copy()

print(f'   Men  : train={len(m_train_cut):,}  val={len(m_val_cut):,}')
print(f'   Women: train={len(w_train_cut):,}  val={len(w_val_cut):,}')

M_FEAT_COLS = shadow_feature_selection(m_all_df, FEAT_COLS, "Men's",   CFG['val_seasons'], threshold=0.45)
W_FEAT_COLS = shadow_feature_selection(w_all_df, FEAT_COLS, "Women's", CFG['val_seasons'], threshold=0.45)

_TRAIN_OK = False
try:
    m_models = train_gender_model(m_train_cut, m_val_cut, M_FEAT_COLS, "Men's")
    w_models = train_gender_model(w_train_cut, w_val_cut, W_FEAT_COLS, "Women's")
    _TRAIN_OK = True
except Exception as _e:
    print(f'\n Training crashed: {_e}')
    import traceback; traceback.print_exc()

if not _TRAIN_OK:
    def _elo_prob(t1, t2, season, elo_lu, init=CFG['elo_init']):
        e1 = elo_lu.get((season, t1), init)
        e2 = elo_lu.get((season, t2), init)
        return float(np.clip(1.0 / (1.0 + 10**((e2-e1)/400)), 0.025, 0.975))
    for sub_path, sub_df_src in [
        (OUTPUT / 'submission_stage1_v3.csv', sub1),
        (OUTPUT / 'submission_stage2_v3.csv', sub2),
    ]:
        rows = []
        for _, r in sub_df_src.iterrows():
            parts = str(r['ID']).split('_')
            season, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
            lu = w_elo_lu if t1 >= 3000 else m_elo_lu
            rows.append({'ID': r['ID'], 'Pred': _elo_prob(t1, t2, season, lu)})
        pd.DataFrame(rows).to_csv(sub_path, index=False)
        print(f'   Fallback saved: {sub_path.name}')
    raise SystemExit(0)

combined_val_preds = np.concatenate([m_models['val_preds_cal'], w_models['val_preds_cal']])
combined_val_y     = np.concatenate([m_models['y_val'], w_models['y_val']])
overall_cal_brier  = brier_score_loss(combined_val_y, combined_val_preds)

print(f'\n   Men\'s   calibrated Brier : {m_models["cal_brier"]:.5f}')
print(f"   Women's calibrated Brier : {w_models['cal_brier']:.5f}")
print(f'   Overall calibrated Brier : {overall_cal_brier:.5f}')

ens_val_preds = np.concatenate([m_models['val_preds_ens'], w_models['val_preds_ens']])
cal_val_preds = combined_val_preds
y_VAL         = combined_val_y
lgb_cv_model  = m_models['lgb_cv_model']
xgb_cv_model  = m_models['xgb_cv_model']
ALL_FEAT_COLS = M_FEAT_COLS


[8/10] Training models ...
   GPU : True  (Tesla P100-PCIE-16GB)
   Men  : train=2,112  val=402
   Women: train=2,024  val=402
   Men's feature selection on 1,444 games ...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   Men's selected 28 / 154 features
   Women's feature selection on 1,386 games ...
   Women's selected 27 / 154 features

   Men's: train=2,112  val=402
   Tuning Men's hyperparameters (Optuna TPE) ...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


   Men's LGB best CV Brier=0.12326
   Men's XGB best CV Brier=0.14420
   Men's CAT best CV Brier=0.03922
[200]	valid_0's binary_logloss: 0.30025
[400]	valid_0's binary_logloss: 0.256369
   Men's LightGBM  Brier=0.08073  iter=389
   Men's XGBoost   Brier=0.10334  iter=903
   Men's CatBoost  Brier=0.02615  iter=159
   Men's LogReg    Brier=0.14867
   Men's MLP       Brier=0.04991
   Men's Ensemble  Brier=0.04193  (LGB=0.14 XGB=0.11 CAT=0.44 LR=0.08 MLP=0.23)
   Men's Calibrated Brier=0.02805  (Δ=-0.01388)

   Women's: train=2,024  val=402
   Tuning Women's hyperparameters (Optuna TPE) ...
   Women's LGB best CV Brier=0.13351
   Women's XGB best CV Brier=0.13247
   Women's CAT best CV Brier=0.03275
[200]	valid_0's binary_logloss: 0.358955
   Women's LightGBM  Brier=0.11727  iter=205
   Women's XGBoost   Brier=0.10002  iter=1454
   Women's CatBoost  Brier=0.03873  iter=467
   Women's LogReg    Brier=0.13745
   Women's MLP       Brier=0.04981
   Women's Ensemble  Brier=0.05503  (LGB=0.12 XG

In [10]:
print(f'\n[9/10] Generating submissions ...')

MODELS = {'men': m_models, 'women': w_models}

def predict_batch(matchups_df, models_dict, clip_lo=CFG['clip_lo'], clip_hi=CFG['clip_hi']):
    preds = np.full(len(matchups_df), np.nan)
    for gender_flag, key in [(0,'men'), (1,'women')]:
        mask = (matchups_df['IsWomen'].values == gender_flag)
        if mask.sum() == 0:
            continue
        mdl       = models_dict[key]
        feat_cols = mdl['feat_cols']
        sub       = matchups_df.loc[mask, feat_cols]

        p_lgb = mdl['lgb_model'].predict(sub)
        p_xgb = mdl['xgb_model'].predict(xgb.DMatrix(sub))
        p_cat = mdl['cat_model'].predict_proba(sub)[:, 1]

        sub_lr = mdl['lr_sca'].transform(mdl['lr_imp'].transform(sub))
        p_lr   = mdl['lr_model'].predict_proba(sub_lr)[:, 1]
        p_mlp  = mdl['mlp_model'].predict_proba(sub_lr)[:, 1]

        ens   = (mdl['w_lgb'] * p_lgb + mdl['w_xgb'] * p_xgb +
                 mdl['w_cat'] * p_cat + mdl['w_lr']  * p_lr  +
                 mdl['w_mlp'] * p_mlp)
        cal   = mdl['iso_cal'].predict(ens)
        preds[mask] = cal

    return np.clip(preds, clip_lo, clip_hi)


def build_features_for_submission(sub_df, label):
    CHUNK  = 5_000
    chunks = []
    n      = len(sub_df)
    for start in tqdm(range(0, n, CHUNK), desc=f'   {label}'):
        chunk      = sub_df.iloc[start:start + CHUNK]
        chunk_rows = []
        for _, r in chunk.iterrows():
            s, t1, t2 = r['Season'], int(r['T1']), int(r['T2'])
            is_women  = (t1 >= 3000)
            feats     = build_matchup_features(s, t1, t2, is_women)
            feats['Season']  = s
            feats['IsWomen'] = int(is_women)
            feats['ID']      = r['ID']
            chunk_rows.append(feats)
        chunks.append(pd.DataFrame(chunk_rows))
        del chunk_rows
    result = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    return result

s1_path   = OUTPUT / 'submission_stage1_v3.csv'
s2_path   = OUTPUT / 'submission_stage2_v3.csv'
comb_path = OUTPUT / 'submission_combined_v3.csv'

try:
    print(f'   Building Stage-1 features ({len(sub1):,} matchups) ...')
    s1_feat_df  = build_features_for_submission(sub1, 'Stage-1')
    s1_preds    = predict_batch(s1_feat_df, MODELS)
    s1_sub      = pd.DataFrame({'ID': s1_feat_df['ID'], 'Pred': s1_preds})
    s1_is_women = s1_feat_df['IsWomen'].values.copy()
    del s1_feat_df; gc.collect()

    print(f'   Building Stage-2 features ({len(sub2):,} matchups) ...')
    s2_feat_df  = build_features_for_submission(sub2, 'Stage-2')
    s2_preds    = predict_batch(s2_feat_df, MODELS)
    s2_sub      = pd.DataFrame({'ID': s2_feat_df['ID'], 'Pred': s2_preds})
    del s2_feat_df; gc.collect()

    combined = pd.concat([s1_sub, s2_sub], ignore_index=True)
    s1_sub.to_csv(s1_path,    index=False)
    s2_sub.to_csv(s2_path,    index=False)
    combined.to_csv(comb_path, index=False)

    print(f'\n   Submissions saved:')
    print(f'      {s1_path.name}  {len(s1_sub):,} rows  <- SUBMIT THIS')
    print(f'      {s2_path.name}  {len(s2_sub):,} rows  <- after Selection Sunday')
    print(f'   Pred mean: {s1_preds.mean():.4f}  std: {s1_preds.std():.4f}  '
          f'range: [{s1_preds.min():.4f}, {s1_preds.max():.4f}]')

except Exception as _pred_e:
    print(f'\n Prediction crashed: {_pred_e}')
    import traceback; traceback.print_exc()
    def _elo_prob_fb(t1, t2, season):
        lu = w_elo_lu if t1 >= 3000 else m_elo_lu
        e1, e2 = lu.get((season,t1), CFG['elo_init']), lu.get((season,t2), CFG['elo_init'])
        return float(np.clip(1.0/(1.0+10**((e2-e1)/400)), 0.025, 0.975))
    for path, src in [(s1_path, sub1), (s2_path, sub2)]:
        rows = []
        for _, r in src.iterrows():
            p = str(r['ID']).split('_')
            rows.append({'ID': r['ID'], 'Pred': _elo_prob_fb(int(p[1]), int(p[2]), int(p[0]))})
        pd.DataFrame(rows).to_csv(path, index=False)
    pd.concat([pd.read_csv(s1_path), pd.read_csv(s2_path)]).to_csv(comb_path, index=False)
    s1_sub = pd.read_csv(s1_path)
    s2_sub = pd.read_csv(s2_path)
    s1_preds = s1_sub['Pred'].values
    s1_is_women = np.zeros(len(s1_sub))



[9/10] Generating submissions ...
   Building Stage-1 features (519,144 matchups) ...


   Stage-1: 100%|██████████| 104/104 [01:11<00:00,  1.45it/s]


   Building Stage-2 features (132,133 matchups) ...


   Stage-2: 100%|██████████| 27/27 [00:17<00:00,  1.53it/s]



   Submissions saved:
      submission_stage1_v3.csv  519,144 rows  <- SUBMIT THIS
      submission_stage2_v3.csv  132,133 rows  <- after Selection Sunday
   Pred mean: 0.4595  std: 0.3961  range: [0.0100, 0.9900]


In [11]:
print('\n[10/10] Generating diagnostic plots ...')

fig = plt.figure(figsize=(24, 18), facecolor='#0f172a')
fig.suptitle('March Mania 2026  —  Model Diagnostics v3.0',
             fontsize=16, fontweight='bold', color='#f8fafc', y=0.98)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
C   = ['#38bdf8','#f472b6','#34d399','#fb923c','#a78bfa','#fbbf24']
axk = dict(facecolor='#1e293b')

ax = fig.add_subplot(gs[0, 0], **axk)
frac, mean_pred = calibration_curve(y_VAL, ens_val_preds, n_bins=10)
frac_c, mean_c  = calibration_curve(y_VAL, cal_val_preds, n_bins=10)
ax.plot(mean_pred, frac,   'o--', color=C[0], lw=1.5, ms=5, label='Uncalibrated')
ax.plot(mean_c,    frac_c, 's-',  color=C[1], lw=2,   ms=6, label='Calibrated')
ax.plot([0,1],[0,1], '--', color='#64748b', lw=1, label='Perfect')
ax.set_title('Calibration Curve', color='#f8fafc')
ax.set_xlabel('Mean Predicted Prob', color='#cbd5e1')
ax.set_ylabel('Fraction of Positives', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[0, 1], **axk)
ax.hist(s1_preds[s1_is_women == 0], bins=50, color=C[0], alpha=0.7, density=True, label="Men's")
ax.hist(s1_preds[s1_is_women == 1], bins=50, color=C[1], alpha=0.7, density=True, label="Women's")
ax.set_title('Prediction Distribution (Stage-1)', color='#f8fafc')
ax.set_xlabel('Predicted Probability', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[0, 2], **axk)
fi_lgb = pd.Series(lgb_cv_model.feature_importance(importance_type='gain'),
                   index=ALL_FEAT_COLS).nlargest(20).sort_values()
ax.barh(range(len(fi_lgb)), fi_lgb.values, color=C[2], edgecolor='#475569')
ax.set_yticks(range(len(fi_lgb)))
ax.set_yticklabels(fi_lgb.index, fontsize=7)
ax.set_title('LightGBM Feature Importance (Gain, Top 20)', color='#f8fafc')
ax.tick_params(colors='#94a3b8')

ax = fig.add_subplot(gs[1, 0], **axk)
fi_xgb = pd.Series(xgb_cv_model.get_score(importance_type='gain')).nlargest(20).sort_values()
ax.barh(range(len(fi_xgb)), fi_xgb.values, color=C[3], edgecolor='#475569')
ax.set_yticks(range(len(fi_xgb)))
ax.set_yticklabels(fi_xgb.index, fontsize=7)
ax.set_title('XGBoost Feature Importance (Gain, Top 20)', color='#f8fafc')
ax.tick_params(colors='#94a3b8')

ax = fig.add_subplot(gs[1, 1], **axk)
val_cut2 = pd.concat([
    m_val_cut[['Season','Outcome']].assign(pred=m_models['val_preds_cal'], IsWomen=0),
    w_val_cut[['Season','Outcome']].assign(pred=w_models['val_preds_cal'], IsWomen=1),
], ignore_index=True)
val_cut2['sq_err'] = (val_cut2['pred'] - val_cut2['Outcome']) ** 2
bs_by_season = val_cut2.groupby('Season')['sq_err'].mean()
ax.bar(bs_by_season.index, bs_by_season.values, color=C[4], edgecolor='#475569', width=0.6)
ax.axhline(0.25, color='#fbbf24', ls='--', lw=1.5, label='Random (0.25)')
ax.set_title('Brier Score by Validation Season', color='#f8fafc')
ax.set_xlabel('Season', color='#cbd5e1')
ax.set_ylabel('Brier Score', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=8)
for x, v in zip(bs_by_season.index, bs_by_season.values):
    ax.text(x, v + 0.002, f'{v:.4f}', ha='center', fontsize=8, color='#cbd5e1')

ax = fig.add_subplot(gs[1, 2], **axk)
for label, flag, color in [("Men's", 0, C[0]), ("Women's", 1, C[1])]:
    mask = val_cut2['IsWomen'] == flag
    bs_g = val_cut2[mask].groupby('Season')['sq_err'].mean()
    ax.plot(bs_g.index, bs_g.values, marker='o', lw=2, ms=6, color=color, label=label)
ax.axhline(0.25, color='#fbbf24', ls='--', lw=1.5)
ax.set_title('Brier Score by Gender & Season', color='#f8fafc')
ax.set_xlabel('Season', color='#cbd5e1')
ax.set_ylabel('Brier Score', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=9)

ax = fig.add_subplot(gs[2, 0:2], **axk)
m_2026 = (m_elo_df[m_elo_df['Season'] == m_elo_df['Season'].max()]
            .merge(m_teams[['TeamID','TeamName']], on='TeamID')
            .nlargest(20, 'EloPreTourney')
            .sort_values('EloPreTourney', ascending=True))
ax.barh(range(len(m_2026)), m_2026['EloPreTourney'], color=C[0], edgecolor='#475569', alpha=0.85)
ax.set_yticks(range(len(m_2026)))
ax.set_yticklabels(m_2026['TeamName'], fontsize=8.5)
ax.set_title("Top 20 Men's Teams — 2026 Pre-Tourney Elo", color='#f8fafc')
ax.set_xlabel('Elo Rating', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.axvline(CFG['elo_init'], color='#fbbf24', ls='--', lw=1.5, label='Baseline 1500')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[2, 2], **axk)
w_2026 = (w_elo_df[w_elo_df['Season'] == w_elo_df['Season'].max()]
            .merge(w_teams[['TeamID','TeamName']], on='TeamID')
            .nlargest(20, 'EloPreTourney')
            .sort_values('EloPreTourney', ascending=True))
ax.barh(range(len(w_2026)), w_2026['EloPreTourney'], color=C[1], edgecolor='#475569', alpha=0.85)
ax.set_yticks(range(len(w_2026)))
ax.set_yticklabels(w_2026['TeamName'], fontsize=8.5)
ax.set_title("Top 20 Women's Teams — 2026 Pre-Tourney Elo", color='#f8fafc')
ax.set_xlabel('Elo Rating', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.axvline(CFG['elo_init'], color='#fbbf24', ls='--', lw=1.5)

for a in fig.axes:
    a.set_facecolor('#1e293b')
    for spine in a.spines.values():
        spine.set_edgecolor('#334155')
    a.grid(color='#334155', linestyle='--', alpha=0.4)

plt.savefig(OUTPUT / 'diagnostics_v3.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close()

print(f'\n  Top 20 Men\'s (Elo {m_elo_df["Season"].max()}):')
print(m_2026[::-1][['TeamName','EloPreTourney']].to_string(index=False))
print(f'\n  Top 20 Women\'s (Elo {w_elo_df["Season"].max()}):')
print(w_2026[::-1][['TeamName','EloPreTourney']].to_string(index=False))

elapsed = time.time() - t0
print(f'\n{DIVIDER}')
print(f'  PIPELINE COMPLETE   {elapsed:.1f}s')
print(f'{DIVIDER}')
print(f'  Men\'s   LGB={m_models["lgb_brier"]:.5f}  XGB={m_models["xgb_brier"]:.5f}  '
      f'CAT={m_models["cat_brier"]:.5f}  MLP={m_models["mlp_brier"]:.5f}  '
      f'Ens={m_models["ens_brier"]:.5f}  Cal={m_models["cal_brier"]:.5f}')
print(f'  Women\'s LGB={w_models["lgb_brier"]:.5f}  XGB={w_models["xgb_brier"]:.5f}  '
      f'CAT={w_models["cat_brier"]:.5f}  MLP={w_models["mlp_brier"]:.5f}  '
      f'Ens={w_models["ens_brier"]:.5f}  Cal={w_models["cal_brier"]:.5f}')
print(f'  Overall Cal Brier   : {overall_cal_brier:.5f}')
print(f'  Stage-1 rows        : {len(s1_sub):,}')
print(f'  Stage-2 rows        : {len(s2_sub):,}')
print(f'{DIVIDER}')



[10/10] Generating diagnostic plots ...

  Top 20 Men's (Elo 2026):
    TeamName  EloPreTourney
        Duke    1810.872705
     Florida    1775.411149
     Houston    1772.611889
     Arizona    1769.145222
 Connecticut    1758.147149
     Gonzaga    1757.167182
    Michigan    1751.332869
St Mary's CA    1732.189185
 Michigan St    1727.972059
   St John's    1723.180843
     Alabama    1716.720190
      Purdue    1715.157959
     Iowa St    1704.311356
  Texas Tech    1699.205400
    Illinois    1697.981698
   Tennessee    1694.824227
     Utah St    1682.455376
         VCU    1676.997068
       Akron    1674.243277
    Nebraska    1666.634839

  Top 20 Women's (Elo 2026):
      TeamName  EloPreTourney
   Connecticut    1907.061964
South Carolina    1874.144115
          UCLA    1854.589450
         Texas    1828.817951
           LSU    1800.964935
           TCU    1763.954945
          Iowa    1757.264521
          Duke    1747.920715
       Ohio St    1743.475025
    Louisvill

In [12]:
import pandas as pd, os

src = "/kaggle/working/submission_stage1_v3.csv"
dst = "/kaggle/working/submission.csv"

if os.path.exists(src):
    df = pd.read_csv(src)[['ID','Pred']]
    df.to_csv(dst, index=False)
    print("submission.csv created")
    print("Rows:", len(df))
    print(df.head())
else:
    print("submission_stage1_v3.csv not found")


submission.csv created
Rows: 519144
               ID      Pred
0  2022_1101_1102  0.990000
1  2022_1101_1103  0.736842
2  2022_1101_1104  0.150000
3  2022_1101_1105  0.990000
4  2022_1101_1106  0.990000
